In [26]:
# Install Packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats


In [27]:
# Load in tables
athletes = pd.read_csv("../data/athlete_events.csv")
city_coords = pd.read_csv("../data/city_coords.csv")
country_coords = pd.read_csv("../data/country_coords.csv")
noc_regions = pd.read_csv("../data/noc_regions.csv")

In [37]:
# Merge all original tables
df = athletes.merge(
    noc_regions,
    on="NOC",
    how="left"
)
df = df.merge(
    country_coords,
    left_on="region",
    right_on="Country",
    how="left"
)
df = df.rename(columns={
    "Latitude": "country_lat",
    "Longitude": "country_lon"
})
df = df.merge(
    city_coords,
    on="City",
    how="left"
)
df = df.rename(columns={
    "Latitude": "host_lat",
    "Longitude": "host_lon"
})

In [38]:
# Filter out rows before 1950 (no temperature data)
df = df[df["Year"] >= 1950]

# Remove non country NOCs
df = df[~df["NOC"].isin(["IOA", "ROT"])]

In [39]:
country_lookup = (
    df[
        [
            "NOC",
            "Games",
            "Year",
            "Season",
            "country_lat",
            "country_lon"
        ]
    ]
    .drop_duplicates()
)
print(len(country_lookup))
country_lookup.head()

3309


,NOC,Games,Year,Season,country_lat,country_lon
0,CHN,1992 Summer,1992,Summer,35.0,105.00
1,CHN,2012 Summer,2012,Summer,35.0,105.00
4,NED,1988 Winter,1988,Winter,52.5,5.75
6,NED,1992 Winter,1992,Winter,52.5,5.75
8,NED,1994 Winter,1994,Winter,52.5,5.75


In [40]:
import openmeteo_requests

import pandas as pd
import requests_cache
import time
from retry_requests import retry

In [ ]:
country_temp_file = "../data/country_temperature_lookup.csv"
city_temp_file = "../data/city_temperature_lookup.csv"

In [66]:
try:
    existing_temps = pd.read_csv(country_temp_file)
    completed_rows = existing_temps[existing_temps["avg_temp (NOC)"].notna()]
    completed = set(zip(completed_rows["NOC"], completed_rows["Games"]))
    temperature_rows = existing_temps.to_dict("records")
    print(f"Loaded {len(completed)} completed temperature records")

except:
    completed = set()
    temperature_rows = []
    print("No existing temperature file found")

cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

for i, (_, row) in enumerate(country_lookup.iterrows()):
    noc = row["NOC"]
    games = row["Games"]

    if (noc, games) in completed:
        continue

    lat = row["country_lat"]
    lon = row["country_lon"]
    year = int(row["Year"])
    season = row["Season"]

    if pd.isna(lat) or pd.isna(lon):
        continue

    if season == "Summer":
        start_date = f"{year}-07-01"
        end_date = f"{year}-08-31"

    else:
        start_date = f"{year}-02-01"
        end_date = f"{year}-02-28"

    url = "https://climate-api.open-meteo.com/v1/climate"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "models": ["EC_Earth3P_HR"],
        "daily": "temperature_2m_mean"
    }

    try:
        responses = openmeteo.weather_api(
            url,
            params=params
        )

        response = responses[0]
        daily = response.Daily()
        temps = daily.Variables(0).ValuesAsNumpy()
        avg_temp = np.mean(temps)

        print(f"{noc} | {games} | {avg_temp:.2f}")

    except Exception as e:
        print(f"FAILED: {noc} | {games}")
        print(e)
        avg_temp = np.nan

    temperature_rows.append({
        "NOC": noc,
        "Games": games,
        "avg_temp (NOC)": avg_temp
    })

    if len(temperature_rows) % 100 == 0:
        pd.DataFrame(temperature_rows).to_csv(
            country_temp_file,
            index=False
        )
        print(f"Saved {len(temperature_rows)} rows")

    time.sleep(0.2)

# Save
temperature_df = pd.DataFrame(temperature_rows)
temperature_df = (
    temperature_df
    .drop_duplicates(
        subset=["NOC", "Games"],
        keep="last"
    )
)
temperature_df.to_csv(
    country_temp_file,
    index=False
)

cache_session.close()

print("Finished")

temperature_df.head()

Loaded 3309 completed temperature records
Finished


,NOC,Games,avg_temp (NOC)
0,CHN,1992 Summer,17.823385
1,CHN,2012 Summer,18.458868
2,NED,1988 Winter,3.166599
3,NED,1992 Winter,5.254099
4,NED,1994 Winter,2.207386


In [62]:
city_lookup = (
    df[
        ["City", "Games", "Year", "Season", "host_lat", "host_lon"]
    ]
    .drop_duplicates()
    .dropna(subset=["host_lat", "host_lon"])
)
print(len(city_lookup))

35


In [65]:
try:
    existing_city_temps = pd.read_csv(city_temp_file)
    completed_rows = existing_city_temps[existing_city_temps["avg_temp (City)"].notna()]
    completed = set(zip(completed_rows["City"], completed_rows["Games"]))
    city_temperature_rows = existing_city_temps.to_dict("records")
    print(f"Loaded {len(completed)} completed city temperature records")
except:
    completed = set()
    city_temperature_rows = []
    print("No existing host temperature file found")

cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

for i, (_, row) in enumerate(city_lookup.iterrows()):
    city = row["City"]
    games = row["Games"]

    if (city, games) in completed:
        continue

    lat = row["host_lat"]
    lon = row["host_lon"]
    year = int(row["Year"])
    season = row["Season"]

    if pd.isna(lat) or pd.isna(lon):
        continue

    if season == "Summer":
        start_date = f"{year}-07-01"
        end_date = f"{year}-08-31"
    else:
        start_date = f"{year}-02-01"
        end_date = f"{year}-02-28"

    url = "https://climate-api.open-meteo.com/v1/climate"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "models": ["EC_Earth3P_HR"],
        "daily": "temperature_2m_mean"
    }

    try:
        responses = openmeteo.weather_api(url, params=params)
        response = responses[0]

        daily = response.Daily()
        temps = daily.Variables(0).ValuesAsNumpy()
        avg_temp = np.mean(temps)

        print(f"{city} | {games} | {avg_temp:.2f}")

    except Exception as e:
        print(f"FAILED: {city} | {games}")
        print(e)
        avg_temp = np.nan

    city_temperature_rows.append({
        "City": city,
        "Games": games,
        "avg_temp (City)": avg_temp
    })

    
    if len(city_temperature_rows) % 50 == 0:
        pd.DataFrame(city_temperature_rows).to_csv(
            city_temp_file,
            index=False
        )
        print(f"Saved {len(city_temperature_rows)} city rows")

    time.sleep(0.2)

# Save
city_temperature_df = pd.DataFrame(city_temperature_rows)

city_temperature_df = city_temperature_df.drop_duplicates(
    subset=["City", "Games"],
    keep="last"
)

city_temperature_df.to_csv(city_temp_file, index=False)

print("Finished")
city_temperature_df.head()

Loaded 35 completed city temperature records
Finished


,City,Games,avg_temp (City)
0,Barcelona,1992 Summer,23.585410
1,London,2012 Summer,16.470438
2,Calgary,1988 Winter,-6.625277
3,Albertville,1992 Winter,2.214772
4,Lillehammer,1994 Winter,-7.322265
